# Liu2024 TWFB + DGFMDRM — pyRiemann Reproduction

**Goal:** Reproduce (or closely approximate) the Liu2024 TWFB+DGFMDRM method using Python and pyRiemann.

**What Liu2024 actually did (from the paper + MATLAB source):**
- The MATLAB file uses freq_bands = {[8,12],[8,20],[8,30],[12,20],[15,20],[15,30],[20,30],[8,15]} and
  picks the best band per 10-fold random split — **not** the 19-band TWFB in Table 3 of the paper.
  The paper's Table 3 describes the full algorithm; the provided MATLAB is a simplified 8-band version.
- Classification is via `fgmdm` = Fisher Geodesic MDM (discriminant geodesic filtering + MDM).
- Time window: 2s of data starting 800 samples before trigger at 500 Hz → 800:2800 → 2000 samples.
  At 500 Hz that is samples 0–4s of the MI period (the trigger marks MI onset at 800 samples in).

**Our Python approximation:**
- MODE `broad_8_30_mdm`: Single 8–30 Hz band, covariances + MDM. Fast sanity check.
- MODE `filterbank_tangent_lda`: All filter-band × time-window combos, tangent-space concat + shrinkage LDA.
- MODE `twfb_inner_selection`: Inner-fold validation selects best band/window combo (leakage-safe).
- MODE `matlab_faithful_attempt`: Matches the MATLAB 8-band set + MDM, random split repeated 10×.

**Deviations from MATLAB:**
1. MATLAB uses raw covariance X'X (not normalised). We use OAS or empirical estimator.
2. MATLAB `fgmdm` = Fisher Geodesic MDM — we approximate with TangentSpace + LDA or plain MDM.
3. MATLAB re-randomises splits every repeat. We use StratifiedShuffleSplit for reproducibility.
4. Paper's 19-band TWFB is in `filterbank_tangent_lda`; MATLAB's 8-band is in `matlab_faithful_attempt`.

## 1. Imports

In [6]:
import os
import re
import sys
import json
import random
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy import signal as sp_signal

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM
from pyriemann.utils.mean import mean_covariance

warnings.filterwarnings('ignore', category=RuntimeWarning)
print(f"numpy {np.__version__}, pandas {pd.__version__}")
try:
    import pyriemann
    print(f"pyriemann {pyriemann.__version__}")
except Exception:
    print("pyriemann not found — install with: pip install pyriemann")

numpy 2.4.3, pandas 3.0.1
pyriemann 0.10


## 2. CONFIG

In [7]:
CONFIG = {
    # --- Paths ---
    "data_root": "../../liu2024_data/liu2024_figshare/sourcedata",  # adjust to your path
    "artifact_root": "../../artifacts/liu2024_twfb_dgfmrdm_pyriemann",

    # --- Dataset ---
    "subjects": "all",           # "all" or list of ints e.g. [1, 2, 3]
    "random_state": 2026,
    "sfreq_raw": 500,

    # --- MI window (seconds, relative to MI onset at t=0) ---
    "mi_window_s": (0.0, 4.0),

    # --- Cross-validation ---
    "n_repeats": 10,
    "test_size": 0.40,           # 40 trials * 0.40 = 16 test, 24 train (matches Liu2024)

    # --- Covariance estimation ---
    "cov_estimator": "oas",      # 'oas', 'lwf', 'scm' (scm = raw X'X like MATLAB)
    "cov_reg_eps": 1e-6,         # added to diagonal to ensure SPD

    # --- Mode ---
    # Options:
    #   'broad_8_30_mdm'         : single band, MDM (fastest)
    #   'filterbank_tangent_lda' : paper's 19 bands x 7 windows, tangent + shrinkage LDA
    #   'twfb_inner_selection'   : inner-fold selects best band+window (leakage-safe)
    #   'matlab_faithful_attempt': 8 MATLAB bands, MDM, matches MATLAB structure
    "mode": "filterbank_tangent_lda",

    # --- Filter banks (paper Table 3 — 19 bands) ---
    "filter_bands_hz": [
        (8, 12), (9, 13), (10, 14), (11, 15), (12, 16),
        (13, 17), (14, 18), (15, 19), (16, 20), (17, 21),
        (18, 22), (19, 23), (20, 24), (21, 25), (22, 26),
        (23, 27), (24, 28), (25, 29), (26, 30),
    ],

    # --- MATLAB 8-band set (for matlab_faithful_attempt mode) ---
    "matlab_bands_hz": [
        (8, 12), (8, 20), (8, 30), (12, 20),
        (15, 20), (15, 30), (20, 30), (8, 15),
    ],

    # --- Time windows (seconds, relative to MI onset) ---
    "time_windows_s": [
        (0.0, 1.0), (0.5, 1.5), (1.0, 2.0), (1.5, 2.5),
        (2.0, 3.0), (2.5, 3.5), (3.0, 4.0),
    ],

    # --- Inner CV for twfb_inner_selection mode ---
    "use_inner_selection": True,
    "inner_cv_splits": 3,

    # --- Classifier ---
    "classifier": "tangent_shrinkage_lda",  # or 'mdm'
}

DATA_ROOT = Path(CONFIG["data_root"])
ARTIFACT_ROOT = Path(CONFIG["artifact_root"])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

SFREQ = int(CONFIG["sfreq_raw"])   # We work at raw 500 Hz for Riemannian (no resampling needed)
MI_START = int(CONFIG["mi_window_s"][0] * SFREQ)
MI_STOP  = int(CONFIG["mi_window_s"][1] * SFREQ)
MI_SAMPLES = MI_STOP - MI_START

np.random.seed(CONFIG["random_state"])
random.seed(CONFIG["random_state"])

print(f"Mode:         {CONFIG['mode']}")
print(f"Data root:    {DATA_ROOT}")
print(f"Artifacts:    {ARTIFACT_ROOT}")
print(f"SFREQ:        {SFREQ} Hz")
print(f"MI window:    {CONFIG['mi_window_s']} → {MI_SAMPLES} samples")
print(f"n_repeats:    {CONFIG['n_repeats']}, test_size: {CONFIG['test_size']}")

Mode:         filterbank_tangent_lda
Data root:    ../../liu2024_data/liu2024_figshare/sourcedata
Artifacts:    ../../artifacts/liu2024_twfb_dgfmrdm_pyriemann
SFREQ:        500 Hz
MI window:    (0.0, 4.0) → 2000 samples
n_repeats:    10, test_size: 0.4


## 3. Liu2024 Channel Constants

In [8]:
# Liu2024 source MAT: trials x 33 channels x 4000 samples at 500 Hz
# Channel 17 (0-indexed) = CPz source reference → drop
# Channels 30, 31 = EOG → drop
# Channel 32 = marker → drop
# Keep 29 EEG channels

SOURCE_EEG_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",          # idx 17 = CPz ref
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
CPZ_IDX = 17
EEG_KEEP_IDX = [i for i in range(30) if i != CPZ_IDX]   # 29 channels
EEG_NAMES = [SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]
N_CHANS = len(EEG_KEEP_IDX)  # 29

# NOTE: The MATLAB file uses channel = [1:17 19:30] (1-indexed) which is indices 0:17, 18:30
# in 0-indexed Python = same 29 channels (drops index 17 = CPz). Consistent.

print(f"EEG channels: {N_CHANS}")
print(f"Channel names: {EEG_NAMES}")

EEG channels: 29
Channel names: ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']


## 4. Data Loading

In [9]:
def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    return int(nums[-1]) if nums else None


def _is_mat_struct(value):
    return hasattr(value, "_fieldnames")


def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Some Liu2024 source files expose a top-level `eeg` MATLAB struct instead of
    top-level `rawdata` / `labels`. This walker lets us search nested fields
    without assuming one exact file layout.
    """
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).startswith("__"):
                continue
            name = f"{prefix}.{key}" if prefix else str(key)
            yield name, value
            yield from _walk_mat_object(value, name)
    elif _is_mat_struct(obj):
        for key in obj._fieldnames:
            value = getattr(obj, key)
            name = f"{prefix}.{key}" if prefix else str(key)
            yield name, value
            yield from _walk_mat_object(value, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")


def _find_first_array(mat, min_ndim=1, name_hints=()):
    """Find the best matching ndarray anywhere in a loaded MAT file."""
    candidates = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.ndim >= min_ndim and value.dtype != object:
            score = 0
            lname = name.lower()
            for hint in name_hints:
                if hint in lname:
                    score += 10
            if value.ndim == 3:
                score += 5
            if any(size in (39, 40) for size in value.shape):
                score += 3
            if max(value.shape) >= 3000:
                score += 2
            candidates.append((score, name, value))
    if not candidates:
        return None, None
    _, name, value = sorted(candidates, key=lambda item: item[0], reverse=True)[0]
    return name, value


def _normalize_trials_channels_samples(raw, labels=None):
    """Normalize an array to shape (trials, channels, samples)."""
    arr = np.asarray(raw, dtype=np.float64)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D raw data, got shape {arr.shape}")

    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # Ensure channels land on axis 1 and samples on axis 2.
    if arr.shape[1] == 33 or arr.shape[1] >= 29:
        pass
    elif arr.shape[2] == 33 or arr.shape[2] >= 29:
        arr = arr.transpose(0, 2, 1)
    else:
        # Fall back to treating the largest remaining axis as time.
        time_axis = int(np.argmax(arr.shape[1:]) + 1)
        if time_axis != 2:
            arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize raw data to trials x channels x samples, got {arr.shape}")
    return arr


def load_subject(mat_path):
    """Load one Liu2024 subject. Returns X (40, 29, MI_SAMPLES) float64, y (40,) int."""
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)

    # --- Debug print on very first call ---
    if not hasattr(load_subject, "_debug_done"):
        load_subject._debug_done = True
        keys = [k for k in mat.keys() if not k.startswith("__")]
        print(f"  [debug] MAT top-level keys: {keys}")
        for k in keys:
            v = mat[k]
            shp = getattr(v, "shape", type(v).__name__)
            print(f"    {k}: {shp}")

    # --- Find raw array and labels anywhere in the MAT structure ---
    raw = None
    raw_name = None
    for key in ["rawdata", "data", "eeg", "EEG", "raw"]:
        if key in mat:
            value = mat[key]
            if isinstance(value, np.ndarray) and value.ndim == 3 and value.dtype != object:
                raw = value
                raw_name = key
                break
            if _is_mat_struct(value) or isinstance(value, dict):
                raw_name, raw = _find_first_array(value, min_ndim=3, name_hints=("rawdata", "data", "eeg", "trial"))
                if raw is not None:
                    break
    if raw is None:
        raw_name, raw = _find_first_array(mat, min_ndim=3, name_hints=("rawdata", "data", "eeg", "trial"))
    if raw is None:
        raise ValueError(f"Cannot find 3D raw data in {mat_path}. Keys: {list(mat.keys())}")

    label_name, labels = None, None
    for key in ["labels", "label", "y", "target"]:
        if key in mat:
            value = mat[key]
            if isinstance(value, np.ndarray) and value.dtype != object:
                labels = value
                label_name = key
                break
            if _is_mat_struct(value) or isinstance(value, dict):
                label_name, labels = _find_first_array(value, min_ndim=1, name_hints=("label", "class", "target", "y"))
                if labels is not None:
                    break
    if labels is None:
        label_name, labels = _find_first_array(mat, min_ndim=1, name_hints=("label", "class", "target", "y"))
    if labels is None:
        raise ValueError(f"Cannot find labels in {mat_path}. Keys: {list(mat.keys())}")

    raw = _normalize_trials_channels_samples(raw, labels=labels)
    labels = np.asarray(labels, dtype=int).ravel()

    # --- Normalise axis order to (40, 33, n_samples) ---
    # Find the axis that is exactly 40 (trials)
    trial_ax = next((ax for ax, sz in enumerate(raw.shape) if sz == 40), None)
    if trial_ax is None:
        raise ValueError(f"No axis of size 40 in shape {raw.shape} from {mat_path}")
    raw = np.moveaxis(raw, trial_ax, 0)  # → (40, ?, ?)

    # Find the axis that is 33 (channels)
    if raw.shape[1] == 33:
        pass                              # already (40, 33, n_samples)
    elif raw.shape[2] == 33:
        raw = raw.transpose(0, 2, 1)     # → (40, 33, n_samples)
    else:
        raise ValueError(f"No axis of size 33 in {raw.shape} from {mat_path}")

    n_samples = raw.shape[2]

    # --- Check MI window fits ---
    if MI_STOP > n_samples:
        raise ValueError(
            f"MI window [{MI_START}:{MI_STOP}] exceeds data length {n_samples} "
            f"({n_samples/SFREQ:.2f}s). Reduce mi_window_s in CONFIG.")

    # --- Select 29 EEG channels, crop MI window ---
    X = raw[:, EEG_KEEP_IDX, MI_START:MI_STOP]   # (40, 29, MI_SAMPLES)

    if set(np.unique(labels).tolist()).issubset({1, 2}):
        labels = labels - 1  # map 1/2 → 0/1

    assert X.shape == (40, N_CHANS, MI_SAMPLES), f"Unexpected X shape: {X.shape}"
    assert len(labels) == 40, f"Expected 40 labels, got {len(labels)}"
    assert np.isfinite(X).all(), "Non-finite values in EEG data"

    return X, labels


def find_mat_files(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            f"Data root not found: {root}\n"
            f"  → Update CONFIG['data_root'] to point at your Liu2024 sourcedata folder.")
    files = sorted(root.rglob("*.mat"))
    if not files:
        raise FileNotFoundError(f"No .mat files found under {root}")
    return files


mat_files = find_mat_files(DATA_ROOT)
all_sids  = sorted({subject_id_from_path(f) for f in mat_files
                    if subject_id_from_path(f) is not None})

SUBJECT_IDS = all_sids if CONFIG["subjects"] == "all" \
              else sorted(int(s) for s in CONFIG["subjects"] )

sid_to_path = {subject_id_from_path(f): f
               for f in mat_files if subject_id_from_path(f) in SUBJECT_IDS}

print(f"Found {len(mat_files)} .mat files")
print(f"Subject IDs ({len(SUBJECT_IDS)}): "
      f"{SUBJECT_IDS[:10]}{'...' if len(SUBJECT_IDS) > 10 else ''}")

# --- Sanity-check first subject ---
print("\nSanity-checking subject", SUBJECT_IDS[0], "...")
_X0, _y0 = load_subject(sid_to_path[SUBJECT_IDS[0]])
print(f"  X shape: {_X0.shape}  |  y: {np.unique(_y0, return_counts=True)}")
del _X0, _y0
print("Data loading OK.")

Found 50 .mat files
Subject IDs (50): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...

Sanity-checking subject 1 ...
  [debug] MAT top-level keys: ['eeg']
    eeg: mat_struct
  X shape: (40, 29, 2000)  |  y: (array([0, 1]), array([20, 20]))
Data loading OK.


## 5. Signal Processing Utilities

**Leakage note:** Bandpass filtering is applied per-trial to the raw signal. This is safe — the filter is a fixed transform (no data-driven parameters). Covariance means are computed on training-fold trials only.

In [10]:
def bandpass_trial(X_trial, low, high, sfreq, order=4):
    """Zero-phase Butterworth bandpass for one trial (n_chans x n_times)."""
    nyq = sfreq / 2.0
    b, a = sp_signal.butter(order, [low / nyq, high / nyq], btype="band")
    return sp_signal.filtfilt(b, a, X_trial, axis=-1)


def bandpass_all(X, low, high, sfreq=500, order=4):
    """Bandpass all trials. X: (n_trials, n_chans, n_times)."""
    out = np.empty_like(X)
    for i in range(len(X)):
        out[i] = bandpass_trial(X[i], low, high, sfreq, order)
    return out


def window_trials(X, t_start, t_stop, sfreq=500):
    """Crop trials to a time window (in seconds relative to MI_START)."""
    s0 = int(t_start * sfreq)
    s1 = int(t_stop * sfreq)
    return X[:, :, s0:s1]


def compute_covs(X, estimator="oas", reg_eps=1e-6):
    """
    Compute SPD covariance matrices.
    X: (n_trials, n_chans, n_times)
    Returns: (n_trials, n_chans, n_chans)
    """
    cov_obj = Covariances(estimator=estimator)
    C = cov_obj.fit_transform(X.astype(np.float64))
    # Regularise: add eps * I to guarantee SPD
    if reg_eps > 0:
        I = np.eye(C.shape[-1])
        C = C + reg_eps * I[np.newaxis]
    assert np.all(np.isfinite(C)), "Non-finite covariance matrices"
    return C


# Quick shape smoke test
_x = np.random.randn(4, 29, 500).astype(np.float64)
_xf = bandpass_all(_x, 8, 30, sfreq=500)
_c = compute_covs(_xf, "oas")
assert _c.shape == (4, 29, 29), f"Cov shape wrong: {_c.shape}"
print("Signal processing utilities OK — cov shape:", _c.shape)

Signal processing utilities OK — cov shape: (4, 29, 29)


## 6. Classifier Pipelines

In [11]:
def make_clf(mode="tangent_shrinkage_lda", metric="riemann"):
    """Return a fitted-on-covs classifier."""
    if mode == "mdm":
        return MDM(metric=metric)
    elif mode == "tangent_shrinkage_lda":
        # TangentSpace projects SPD covs to Euclidean tangent vectors,
        # then shrinkage LDA classifies. This approximates DGFMDRM.
        return Pipeline([
            ("ts", TangentSpace(metric=metric)),
            ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
        ])
    else:
        raise ValueError(f"Unknown classifier mode: {mode}")


def fit_predict_covs(C_train, y_train, C_test, clf_mode, metric="riemann"):
    """Fit classifier on training covs and predict test covs."""
    assert len(np.unique(y_train)) == 2, "Expected binary labels"
    clf = make_clf(clf_mode, metric)
    clf.fit(C_train, y_train)
    return clf.predict(C_test)


print("Classifier pipelines defined.")

Classifier pipelines defined.


## 7. Feature Extraction Strategies per Mode

In [12]:
def extract_features_broad(X_train, X_test, cfg):
    """8–30 Hz, full MI window. Returns covariance matrices."""
    Xf_tr = bandpass_all(X_train, 8, 30, SFREQ)
    Xf_te = bandpass_all(X_test,  8, 30, SFREQ)
    return (compute_covs(Xf_tr, cfg["cov_estimator"], cfg["cov_reg_eps"]),
            compute_covs(Xf_te, cfg["cov_estimator"], cfg["cov_reg_eps"]))


def extract_features_filterbank_concat(X_train, X_test, cfg, bands, windows):
    """
    All (band, window) combos → tangent-space vectors concatenated.
    TangentSpace reference fitted on training covs only.
    Returns feat_train (n_train, D), feat_test (n_test, D).
    """
    feats_tr, feats_te = [], []
    for (low, high) in bands:
        Xf_tr = bandpass_all(X_train, low, high, SFREQ)
        Xf_te = bandpass_all(X_test,  low, high, SFREQ)
        for (t0, t1) in windows:
            Xw_tr = window_trials(Xf_tr, t0, t1, SFREQ)
            Xw_te = window_trials(Xf_te, t0, t1, SFREQ)
            C_tr  = compute_covs(Xw_tr, cfg["cov_estimator"], cfg["cov_reg_eps"])
            C_te  = compute_covs(Xw_te, cfg["cov_estimator"], cfg["cov_reg_eps"])
            ts    = TangentSpace(metric="riemann")
            feats_tr.append(ts.fit_transform(C_tr))
            feats_te.append(ts.transform(C_te))
    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)


def select_best_combo_inner(X_train, y_train, cfg, bands, windows, n_inner=3):
    """Inner-fold selection of best (band, window). Uses training data only."""
    inner_cv = StratifiedKFold(n_splits=n_inner, shuffle=True,
                               random_state=cfg["random_state"])
    best_acc, best_combo = -1, (bands[0], windows[0])
    for (low, high) in bands:
        Xf = bandpass_all(X_train, low, high, SFREQ)
        for (t0, t1) in windows:
            Xw = window_trials(Xf, t0, t1, SFREQ)
            accs = []
            for tr_i, va_i in inner_cv.split(Xw, y_train):
                C_tr = compute_covs(Xw[tr_i], cfg["cov_estimator"], cfg["cov_reg_eps"])
                C_va = compute_covs(Xw[va_i], cfg["cov_estimator"], cfg["cov_reg_eps"])
                try:
                    p = fit_predict_covs(C_tr, y_train[tr_i], C_va, cfg["classifier"])
                    accs.append(balanced_accuracy_score(y_train[va_i], p))
                except Exception:
                    accs.append(0.5)
            mean_acc = float(np.mean(accs))
            if mean_acc > best_acc:
                best_acc  = mean_acc
                best_combo = ((low, high), (t0, t1))
    return best_combo[0], best_combo[1], best_acc


print("Feature extraction strategies defined.")


Feature extraction strategies defined.


## 8. Per-Subject Cross-Validation Runner

**Leakage controls enforced here:**
- Bandpass filter is a fixed transform (no fitting on data) → safe to apply before splitting.
- Covariance matrices are computed per split, per fold.
- TangentSpace reference point is fitted on training covs only, then applied to test covs.
- Inner band/window selection (when enabled) uses only training data.
- Classifier is fitted on training data only.

In [13]:
def collapse_diagnostics(y_pred, n_classes=2):
    counts   = np.bincount(y_pred, minlength=n_classes)
    dominant = counts.max() / counts.sum() if counts.sum() > 0 else 1.0
    return {
        "collapse_flag":  bool(dominant > 0.95),
        "collapse_ratio": float(dominant),
        "pred_counts":    counts.tolist(),
    }


def run_subject(sid, X, y, cfg):
    """Run repeated stratified splits for one subject."""
    mode    = cfg["mode"]
    bands   = cfg["filter_bands_hz"] if mode != "matlab_faithful_attempt" else cfg["matlab_bands_hz"]
    windows = cfg["time_windows_s"]

    sss = StratifiedShuffleSplit(
        n_splits=cfg["n_repeats"],
        test_size=cfg["test_size"],
        random_state=cfg["random_state"],
    )

    fold_results = []
    for fold_idx, (tr_idx, te_idx) in enumerate(sss.split(X, y)):
        X_train, X_test = X[tr_idx], X[te_idx]
        y_train, y_test = y[tr_idx], y[te_idx]

        if len(np.unique(y_train)) < 2:
            print(f"  Sub {sid} fold {fold_idx}: only one class in train, skipping")
            continue

        selected_band, selected_window = None, None

        try:
            if mode == "broad_8_30_mdm":
                C_tr, C_te = extract_features_broad(X_train, X_test, cfg)
                y_pred = fit_predict_covs(C_tr, y_train, C_te, "mdm")

            elif mode == "filterbank_tangent_lda":
                feat_tr, feat_te = extract_features_filterbank_concat(
                    X_train, X_test, cfg, bands, windows)
                # Guard against n_samples < n_features for LDA
                n_components = min(feat_tr.shape[1], feat_tr.shape[0] - 1)
                from sklearn.decomposition import PCA
                if feat_tr.shape[1] > feat_tr.shape[0] - 1:
                    pca = PCA(n_components=n_components, random_state=cfg["random_state"])
                    feat_tr = pca.fit_transform(feat_tr)
                    feat_te = pca.transform(feat_te)
                lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
                lda.fit(feat_tr, y_train)
                y_pred = lda.predict(feat_te)

            elif mode == "twfb_inner_selection":
                best_band, best_window, _ = select_best_combo_inner(
                    X_train, y_train, cfg, bands, windows, cfg["inner_cv_splits"])
                selected_band, selected_window = best_band, best_window
                Xf_tr = bandpass_all(X_train, *best_band, SFREQ)
                Xf_te = bandpass_all(X_test,  *best_band, SFREQ)
                Xw_tr = window_trials(Xf_tr, *best_window, SFREQ)
                Xw_te = window_trials(Xf_te, *best_window, SFREQ)
                C_tr  = compute_covs(Xw_tr, cfg["cov_estimator"], cfg["cov_reg_eps"])
                C_te  = compute_covs(Xw_te, cfg["cov_estimator"], cfg["cov_reg_eps"])
                y_pred = fit_predict_covs(C_tr, y_train, C_te, cfg["classifier"])

            elif mode == "matlab_faithful_attempt":
                # Mirrors MATLAB: try all 8 bands, pick best on this fold's test set.
                # NOTE: this has test leakage (same as original MATLAB). Flagged clearly.
                best_acc_t, best_pred = -1, None
                for (low, high) in bands:
                    Xf_tr = bandpass_all(X_train, low, high, SFREQ)
                    Xf_te = bandpass_all(X_test,  low, high, SFREQ)
                    C_tr  = compute_covs(Xf_tr, "scm", cfg["cov_reg_eps"])
                    C_te  = compute_covs(Xf_te, "scm", cfg["cov_reg_eps"])
                    try:
                        preds = fit_predict_covs(C_tr, y_train, C_te, "mdm")
                        acc   = accuracy_score(y_test, preds)
                        if acc > best_acc_t:
                            best_acc_t  = acc
                            best_pred   = preds
                            selected_band = (low, high)
                    except Exception:
                        pass
                y_pred = best_pred if best_pred is not None \
                         else np.full(len(y_test), int(np.bincount(y_train).argmax()))

            else:
                raise ValueError(f"Unknown mode: {mode}")

        except Exception as exc:
            import traceback
            print(f"  Sub {sid} fold {fold_idx} ERROR: {exc}")
            traceback.print_exc()
            y_pred = np.full(len(y_test), int(np.bincount(y_train).argmax()))

        acc  = float(accuracy_score(y_test, y_pred))
        bacc = float(balanced_accuracy_score(y_test, y_pred))
        cm   = confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
        diag = collapse_diagnostics(y_pred)

        cm_arr       = np.array(cm)
        left_recall  = (cm_arr[0, 0] / cm_arr[0].sum()
                        if cm_arr[0].sum() > 0 else float("nan"))
        right_recall = (cm_arr[1, 1] / cm_arr[1].sum()
                        if cm_arr[1].sum() > 0 else float("nan"))

        fold_results.append({
            "subject_id":        sid,
            "fold_id":           fold_idx,
            "accuracy":          acc,
            "balanced_accuracy": bacc,
            "left_recall":       float(left_recall),
            "right_recall":      float(right_recall),
            "confusion_matrix":  cm,
            "collapse_flag":     diag["collapse_flag"],
            "collapse_ratio":    diag["collapse_ratio"],
            "pred_counts":       diag["pred_counts"],
            "n_train":           int(len(y_train)),
            "n_test":            int(len(y_test)),
            "selected_band":     str(selected_band),
            "selected_window":   str(selected_window),
        })

    return fold_results


print("Subject runner defined.")


Subject runner defined.


## 9. Run All Subjects

In [14]:
ALL_FOLD_RESULTS = []
SUBJECT_SUMMARIES = []

print(f"Running {len(SUBJECT_IDS)} subjects | mode={CONFIG['mode']}")
print("=" * 60)

for sid in SUBJECT_IDS:
    mat_path = sid_to_path.get(sid)
    if mat_path is None:
        print(f"  Sub {sid:02d}: no file found, skipping")
        continue

    try:
        X, y = load_subject(mat_path)
    except Exception as exc:
        print(f"  Sub {sid:02d}: load error — {exc}")
        continue

    fold_res = run_subject(sid, X, y, CONFIG)
    ALL_FOLD_RESULTS.extend(fold_res)

    accs  = [r["accuracy"] for r in fold_res]
    baccs = [r["balanced_accuracy"] for r in fold_res]
    collapses = sum(1 for r in fold_res if r["collapse_flag"])
    mean_bacc = np.mean(baccs)

    SUBJECT_SUMMARIES.append({
        "subject_id":             sid,
        "mean_accuracy":          float(np.mean(accs)),
        "std_accuracy":           float(np.std(accs)),
        "mean_balanced_accuracy": float(mean_bacc),
        "std_balanced_accuracy":  float(np.std(baccs)),
        "n_folds":                len(fold_res),
        "n_collapsed_folds":      collapses,
    })

    print(f"  Sub {sid:02d}: bal_acc={mean_bacc*100:.1f}% ± {np.std(baccs)*100:.1f}%  "
          f"collapse={collapses}/{len(fold_res)}")

print("=" * 60)
print(f"Done. Total folds: {len(ALL_FOLD_RESULTS)}")

Running 50 subjects | mode=filterbank_tangent_lda
  Sub 01: bal_acc=40.6% ± 7.5%  collapse=0/10
  Sub 02: bal_acc=35.6% ± 9.3%  collapse=0/10
  Sub 03: bal_acc=49.4% ± 6.5%  collapse=0/10
  Sub 04: bal_acc=50.0% ± 11.5%  collapse=0/10
  Sub 05: bal_acc=59.4% ± 12.3%  collapse=0/10
  Sub 06: bal_acc=39.4% ± 8.4%  collapse=0/10
  Sub 07: bal_acc=60.0% ± 9.4%  collapse=0/10
  Sub 08: bal_acc=50.6% ± 14.1%  collapse=0/10
  Sub 09: bal_acc=56.2% ± 9.3%  collapse=0/10
  Sub 10: bal_acc=48.8% ± 10.0%  collapse=0/10
  Sub 11: bal_acc=48.1% ± 10.1%  collapse=0/10
  Sub 12: bal_acc=48.1% ± 8.9%  collapse=0/10
  Sub 13: bal_acc=46.9% ± 9.4%  collapse=0/10
  Sub 14: bal_acc=53.8% ± 9.8%  collapse=0/10
  Sub 15: bal_acc=48.8% ± 6.1%  collapse=0/10
  Sub 16: bal_acc=41.9% ± 8.4%  collapse=0/10
  Sub 17: bal_acc=49.4% ± 9.0%  collapse=0/10
  Sub 18: bal_acc=49.4% ± 11.0%  collapse=0/10
  Sub 19: bal_acc=50.6% ± 9.0%  collapse=0/10
  Sub 20: bal_acc=60.6% ± 10.8%  collapse=0/10
  Sub 21: bal_acc=39.4%

## 10. Aggregate Results

In [15]:
if not ALL_FOLD_RESULTS:
    raise RuntimeError(
        "ALL_FOLD_RESULTS is empty — no subjects completed successfully.\n"
        "Check the output above for load errors or exceptions.\n"
        "Common causes:\n"
        "  1. CONFIG['data_root'] path is wrong.\n"
        "  2. The .mat files are the processed .edf versions (wrong folder).\n"
        "  3. MI window in CONFIG exceeds the actual trial length."
    )

fold_df    = pd.DataFrame(ALL_FOLD_RESULTS)
subject_df = pd.DataFrame(SUBJECT_SUMMARIES)

all_baccs   = fold_df["balanced_accuracy"].values
all_accs    = fold_df["accuracy"].values
n_collapsed = int(fold_df["collapse_flag"].sum())

cm_total = np.zeros((2, 2), dtype=int)
for row in ALL_FOLD_RESULTS:
    cm_total += np.array(row["confusion_matrix"])

left_recall_agg  = cm_total[0, 0] / cm_total[0].sum() if cm_total[0].sum() > 0 else float("nan")
right_recall_agg = cm_total[1, 1] / cm_total[1].sum() if cm_total[1].sum() > 0 else float("nan")

global_summary = {
    "mode":                   CONFIG["mode"],
    "n_subjects":             len(SUBJECT_SUMMARIES),
    "n_folds_total":          len(ALL_FOLD_RESULTS),
    "mean_accuracy":          float(np.mean(all_accs)),
    "std_accuracy":           float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_baccs)),
    "std_balanced_accuracy":  float(np.std(all_baccs)),
    "n_collapsed_folds":      n_collapsed,
    "collapse_rate":          float(n_collapsed / len(ALL_FOLD_RESULTS)),
    "left_recall_agg":        float(left_recall_agg),
    "right_recall_agg":       float(right_recall_agg),
    "confusion_matrix":       cm_total.tolist(),
}

print("=" * 60)
print(f"GLOBAL RESULTS — mode={CONFIG['mode']}")
print(f"  Mean Balanced Accuracy: {global_summary['mean_balanced_accuracy']*100:.2f}% "
      f"± {global_summary['std_balanced_accuracy']*100:.2f}%")
print(f"  Mean Accuracy:          {global_summary['mean_accuracy']*100:.2f}% "
      f"± {global_summary['std_accuracy']*100:.2f}%")
print(f"  Left recall:  {left_recall_agg*100:.1f}%  |  Right recall: {right_recall_agg*100:.1f}%")
print(f"  Collapsed folds: {n_collapsed}/{len(ALL_FOLD_RESULTS)} ({global_summary['collapse_rate']*100:.1f}%)")
print(f"  Aggregated CM (rows=true, cols=pred, 0=Left 1=Right):")
print(f"    {cm_total}")
print("=" * 60)


GLOBAL RESULTS — mode=filterbank_tangent_lda
  Mean Balanced Accuracy: 51.95% ± 14.28%
  Mean Accuracy:          51.95% ± 14.28%
  Left recall:  52.3%  |  Right recall: 51.6%
  Collapsed folds: 0/500 (0.0%)
  Aggregated CM (rows=true, cols=pred, 0=Left 1=Right):
    [[2091 1909]
 [1935 2065]]


## 11. Save Artifacts

In [16]:
fold_df.to_csv(ARTIFACT_ROOT / "fold_results.csv", index=False)
subject_df.to_csv(ARTIFACT_ROOT / "subject_summary.csv", index=False)

with open(ARTIFACT_ROOT / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=2)

with open(ARTIFACT_ROOT / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print(f"Saved fold_results.csv ({len(fold_df)} rows)")
print(f"Saved subject_summary.csv ({len(subject_df)} rows)")
print(f"Saved global_summary.json")

Saved fold_results.csv (500 rows)
Saved subject_summary.csv (50 rows)
Saved global_summary.json


## 12. Plots

In [17]:
# --- Confusion matrix ---
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm_total, cmap="Blues", aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred Left", "Pred Right"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True Left", "True Right"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_total[i, j]), ha="center", va="center", fontsize=12)
ax.set_title(f"Aggregated CM — mode={CONFIG['mode']}")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix.png")

# --- Per-subject balanced accuracy ---
fig, ax = plt.subplots(figsize=(max(8, len(subject_df) * 0.4), 4))
sids  = subject_df["subject_id"].values
baccs = subject_df["mean_balanced_accuracy"].values * 100
stds  = subject_df["std_balanced_accuracy"].values * 100
ax.bar(sids, baccs, yerr=stds, capsize=3, color="steelblue", alpha=0.8)
ax.axhline(50, color="red", linestyle="--", label="Chance")
ax.axhline(float(np.mean(baccs)), color="orange", linestyle="-", label=f"Mean={np.mean(baccs):.1f}%")
ax.set_xlabel("Subject ID")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title(f"Per-subject balanced accuracy — mode={CONFIG['mode']}")
ax.legend()
ax.set_xticks(sids)
ax.set_xticklabels(sids, rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "subject_accuracy_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved subject_accuracy_plot.png")

Saved confusion_matrix.png
Saved subject_accuracy_plot.png


/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_87462/2592433202.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_87462/2592433202.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
